In [1]:
import os
os.chdir("../")
%pwd

'c:\\Users\\Monalisha\\OneDrive\\Desktop\\Medical Chatbot'

In [73]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

from langchain.text_splitter import RecursiveCharacterTextSplitter

In [3]:
# Extract Data from the PDF file
def load_pdf_file(data):
    loader=DirectoryLoader(data,
                           glob="*.pdf",
                           loader_cls=PyPDFLoader)
    
    documents=loader.load()

    return documents

In [4]:
extracted_data=load_pdf_file(data='Data/')

In [5]:
# extracted_data

In [59]:
# split the Data into text chunks
def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [60]:
text_chunks = text_split(extracted_data)
print("Length of Text chunks ",len(text_chunks))

Length of Text chunks  7939


In [8]:
pip install -U langchain-huggingface


Note: you may need to restart the kernel to use updated packages.


In [61]:
from langchain_huggingface import HuggingFaceEmbeddings


In [66]:
# download the embeddings from huggingface
def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings


In [63]:
embeddings = download_hugging_face_embeddings()


In [64]:
vector = embeddings.embed_query("Hello world!")

print(f"Vector length: {len(vector)}")

Vector length: 384


In [13]:
# vector

In [14]:
pip install pinecone


Note: you may need to restart the kernel to use updated packages.


In [74]:
import os
from dotenv import load_dotenv

load_dotenv()
PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY')
print(PINECONE_API_KEY)  # Should print your real key, not None or the string 'PINECONE_API_KEY'
pc = Pinecone(api_key=PINECONE_API_KEY)


pcsk_3CM99j_T3p4dNhc4idrWTtaZcUb17VY1HbjUE3pb6JVw8MUnS1z61on11zgvsH6eFVYwvi


In [19]:
from dotenv import load_dotenv
load_dotenv()

PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY')
print("Loaded API KEY:", PINECONE_API_KEY)
 

from pinecone import Pinecone
from pinecone import ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = 'medibot'

pc.create_index(
    name=index_name,
    dimension=384,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)


Loaded API KEY: pcsk_3CM99j_T3p4dNhc4idrWTtaZcUb17VY1HbjUE3pb6JVw8MUnS1z61on11zgvsH6eFVYwvi


{
    "name": "medibot",
    "metric": "cosine",
    "host": "medibot-6mezf3h.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

In [17]:
import os
os.environ["PINECONE_API_KEY"]= PINECONE_API_KEY

In [20]:
# Embed each chunk and upsert the embeddings into your Pinecone index.
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embeddings,
)

In [21]:
# Load Existing index
from langchain_pinecone import PineconeVectorStore
 # Embed each chunk and upsert the embeddings into your Pinecone index. 
docsearch = PineconeVectorStore.from_existing_index(
  index_name=index_name,
  embedding=embeddings
)

In [23]:
#docsearch

In [24]:
retriever = docsearch.as_retriever(search_type = "similarity", search_kwargs={"k":3})

In [25]:
retrieved_docs = retriever.invoke("What is Acne?")
retrieved_docs

[Document(id='3eff08cc-3590-467b-a58d-49ffd6fd7161', metadata={'creationdate': '2004-12-18T17:16:32-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:35:04-06:00', 'page': 425.0, 'page_label': '426', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data\\Gale Encyclopedia of Medicine Vol. 2 (C-F).pdf', 'total_pages': 759.0}, page_content='Corticosteriod —A group of synthetic hormones\nthat are used to prevent or reduce inflammation.\nToxic effects may result from rapid withdrawal after\nprolonged use or from continued use of large doses.\nPatch test—A skin test that is done to identify aller-\ngens. A suspected substance is applied to the skin.\nAfter 24–48 hours, if the area is red and swollen,\nthe test is positive for that substance. If no reaction\noccurs, another substance is applied. This is con-'),
 Document(id='cc31a0b0-b433-4068-9608-a8ea128cb330', metadata={'creationdate': '2004-12-18T17:16:32-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:35:04-06:00', 'page': 2

In [27]:
from langchain_ollama import OllamaLLM
llm = OllamaLLM(model="tinyllama")


In [28]:
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=False
)


In [32]:
# Assuming retrieved_docs is your list of retrieved Document objects
context = "\n\n".join([doc.page_content for doc in retrieved_docs])


In [33]:
question = "What is Acne?"

prompt = f"""Use the following context to answer the question.

Context:
{context}

Question: {question}
Answer:"""

answer = llm.invoke(prompt)
print(answer)


Acne is a type of inflammation caused by the overproduction of sebum (oil) by the skin's glands. This leads to pimples or bumps on the face that can be red, tender, and itchy. Acne is a chronic condition that affects an estimated 75% of adults in the United States. It can cause significant social and emotional distress as well as impairing the individual's ability to perform daily activities due to the formation of pimples or bumps.


In [67]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following context to answer the question. "
    "If you don't know the answer, say that "
    "you don't know. Use three sentences maximum and "
    "Keep your answer concise."
    "\n\n"
    "\n{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [68]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [69]:
# Assuming retrieved_docs is your list of retrieved Document objects
context = "\n\n".join([doc.page_content for doc in retrieved_docs])


In [72]:
'''response = rag_chain.invoke({"input": "What is Acne?"})
print(response["answer"])'''

question = "What is Clock?"

prompt = f"""Use the following context to answer the question.

Context:
{context}

Question: {question}
Answer:"""

answer = llm.invoke(prompt)
print(answer)


Cloco is a medication used to treat inflammation. It works by preventing or reducing the effects of inflammatory responses that can result in toxic reactions, such as rapid withdrawal after prolonged use or continued use of large doses. Patch test is an effective skin test used to identify allergens, while a suspected substance is applied to the skin for 24-48 hours if no reaction occurs. If redness and swelling occur, another substance may be applied, but otherwise, the test results are positive for that substance. Immune response is an important component of the immune system that protects against foreign anti-grams (substances the body perceives as potentially dangerous). The immune system can become a chronic and disabling condition that negatively impacts employability and quality of life, which can manifest in various forms of dermatitis.
